# Trabalho de Otimização: Planejamento Ótimo de Rotina de Treinos (MILP)

### Descrição do Problema
Este notebook apresenta um modelo de **Programação Linear Inteira Mista (MILP)** desenvolvido no solver **Gurobi** para prescrever e organizar uma rotina ótima de treinos de musculação ao longo de um horizonte de dias.

O modelo recebe como entrada o catálogo de exercícios, os grupos musculares recrutados, os *Personal Records* (PRs) do praticante, as repetições alvo e a disponibilidade de dias na semana. A partir disso, toma decisões simultâneas em duas escalas de tempo:
1. **Escala Inter-Dias ($i$):** Decide quais grupos musculares e exercícios alocar em cada dia, controlando a curva de recuperação e a fadiga residual entre sessões.
2. **Escala Intra-Treino ($k$):** Decide a ordem exata de execução dos exercícios dentro da sessão e a carga ótima (em kg) a ser utilizada, levando em conta a queda de rendimento gerada pela fadiga acumulada ao longo do treino.

In [ ]:
# Importando as dependências
import gurobipy as gp
from gurobipy import GRB


Essa primeira célula é pra a função que processa os dados de entrada, e coloca no formato que o modelo espera.

In [ ]:


def preprocessar_dados(dados):
  """Calcula o 1RM, o fator de intensidade relativa (I_hat) e a carga máxima

  suportada para as repetições do treino (W_max) usando a fórmula de Epley.
  """
  catalogo = dados["catalogo"]
  musculos = dados["musculos"]
  pref = dados["pref"]

  um_rm, I_hat, W_max, utilidade_base = {}, {}, {}, {}

  for j, info in catalogo.items():
    # 1. Estimativa de 1RM a partir do PR: 1RM = Peso * (1 + Reps / 30)
    um_rm[j] = info["pr_peso"] * (1.0 + info["pr_reps"] / 30.0)

    # 2. Intensidade relativa (% de 1RM) associada às repetições alvo do treino
    I_hat[j] = 1.0 / (1.0 + info["reps_treino"] / 30.0)

    # 3. Carga máxima (100%) em kg para o número de repetições alvo
    W_max[j] = um_rm[j] * I_hat[j]

    # 4. Utilidade hipertrófica ponderada pela preferência muscular do usuário
    utilidade_base[j] = I_hat[j] * sum(
        pref[m] * info["alpha"].get(m, 0.0) for m in musculos
    )

  return um_rm, I_hat, W_max, utilidade_base

### 1. Variável de Decisão: Escolha e Ordem dos Exercícios ($Ex_{i,j,k}$)
Para definir exatamente qual exercício ocupa cada posição do treino, introduzimos a variável binária tridimensional:
$$Ex_{i,j,k} \in \{0, 1\} \quad \forall i \in I, \, j \in J, \, k \in K$$

* **Interpretação:** Assume valor $1$ se o exercício $j$ for executado no dia $i$ na posição (slot de ordem) $k$ do treino, e $0$ caso contrário.
* **Papel no Modelo:** Permite impor exclusividade mútua em cada slot do treino, evitar a repetição do mesmo exercício no mesmo dia e eliminar "buracos" na sequência de execução ($k=1, 2, \dots, K_{\max}$).

In [ ]:
def adicionar_variavel_Ex(model, dias, exercicios, slots):
  Ex = model.addVars(dias, exercicios, slots, vtype=GRB.BINARY, name="Ex")
  return Ex

### 2. Variável de Decisão: Intensidade e Carga Aplicada ($Int_{i,j,k}$)
A variável contínua $Int_{i,j,k}$ modela a fração da carga máxima ($W^{\max}_j$) que o praticante efetivamente utiliza no exercício selecionado:
$$Int_{i,j,k} \in [0, 1] \quad \forall i \in I, \, j \in J, \, k \in K$$

* **Interpretação:** Se $Int_{i,j,k} = 1.0$, o praticante executa a série com $100\%$ da carga máxima possível para aquela faixa de repetições ($W^{\max}_j$).
* **Conversão Direta para Quilogramas (kg):** No pós-processamento, o peso real prescrito na barra/máquina é obtido diretamente por $Peso_{i,j,k} = W^{\max}_j \cdot Int_{i,j,k}$.
* **Linearidade (MILP):** Como $Int_{i,j,k}$ é acoplada linearmente à variável binária $Ex_{i,j,k}$ por desigualdades ($I_{\min} Ex_{i,j,k} \le Int_{i,j,k} \le Ex_{i,j,k}$), evitamos a multiplicação entre variáveis de decisão e mantemos o modelo $100\%$ linear.

In [ ]:
def adicionar_variavel_Int(model, dias, exercicios, slots):
  Int = model.addVars(
      dias, exercicios, slots, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="Int"
  )
  return Int

### 3. Variável de Estado: Fadiga Muscular Inter-Dias ($Fad_{i,m}$)
Diferente de uma fadiga puramente sistêmica, separamos o desgaste por grupo muscular $m \in M$ ao longo dos dias $i \in I$:
$$Fad_{i,m} \in [0, 1] \quad \forall i \in I, \, m \in M$$

* **Interpretação:** Indica o nível de fadiga residual acumulada no grupo muscular $m$ no início do dia $i$ (onde $0$ representa músculo totalmente descansado e $1$ representa exaustão total).
* **Papel no Modelo:** Permite que divisões clássicas de treino (*Push/Pull/Legs*, *Upper/Lower* ou *Full Body*) emerjam naturalmente da otimização, pois enquanto um grupo muscular descansa no dia seguinte, outros grupos com $Fad_{i,m}$ baixa podem ser treinados.

In [ ]:
def adicionar_variavel_Fad(model, dias, musculos):
  Fad = model.addVars(
      dias, musculos, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="Fad"
  )
  return Fad

### 4. Variável de Estado: Fadiga Intra-Treino ($FadTr_{i,k}$)
Para capturar a queda de desempenho dentro de uma mesma sessão de treino à medida que os exercícios são realizados, definimos:
$$FadTr_{i,k} \in [0, 1] \quad \forall i \in I, \, k \in K$$

* **Interpretação:** Representa o cansaço agudo acumulado na sessão do dia $i$ imediatamente antes de iniciar o exercício do slot $k$ (iniciando em $FadTr_{i,1} = 0$).
* **Impacto na Ordem dos Exercícios:** À medida que $FadTr_{i,k}$ cresce nos slots finais ($k=2, 3, \dots$), ela reduz o teto de intensidade disponível ($Int_{i,j,k} \le 1 - FadTr_{i,k}$), forçando o solver a alocar os exercícios compostos mais pesados e prioritários no início do treino.

In [ ]:
def adicionar_variavel_FadTr(model, dias, slots):
  FadTr = model.addVars(
      dias, slots, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="FadTr"
  )
  return FadTr

### 5. Função Objetivo e Restrições do Modelo

#### Função Objetivo
Maximizamos o estímulo hipertrófico total ponderado pela preferência muscular do usuário ($pref_m \cdot \alpha_{j,m}$), pelo fator de carga de 1RM ($\hat{I}_j$) e pela eficiência neuromuscular da posição do exercício na sessão ($w_k$):
$$\max \sum_{i \in I} \sum_{j \in J} \sum_{k \in K} w_k \cdot \hat{I}_j \left( \sum_{m \in M} pref_m \cdot \alpha_{j,m} \right) Int_{i,j,k}$$

#### Restrições Estruturais e Dinâmicas
1. **Disponibilidade e Ordenação Contígua (Bloco A):** Garante no máximo $1$ exercício por slot em dias disponíveis ($\sum_j Ex_{i,j,k} \le Disp_i$), preenchimento sequencial sem buracos ($\sum_j Ex_{i,j,k+1} \le \sum_j Ex_{i,j,k}$) e não repetição do mesmo exercício no dia ($\sum_k Ex_{i,j,k} \le 1$).
2. **Acoplamento Exercício-Intensidade e Degradação Intra-Treino (Bloco B):** 
   $$I_{\min} \cdot Ex_{i,j,k} \le Int_{i,j,k} \le Ex_{i,j,k} \quad \text{e} \quad Int_{i,j,k} \le 1 - FadTr_{i,k}$$
3. **Propagação da Fadiga Intra-Treino (Bloco C):** $FadTr_{i,1} = 0$ e $FadTr_{i,k+1} = FadTr_{i,k} + \sum_j (\gamma_j \hat{I}_j) Int_{i,j,k}$.
4. **Capacidade Diária por Grupo Muscular (Bloco D):** O desgaste local no músculo $m$ somado à fadiga intra-treino respeita a recuperação atual $(1 - Fad_{i,m})$:
   $$\sum_{j,k} (\alpha_{j,m} \hat{I}_j) Int_{i,j,k} + \sum_k FadTr_{i,k} \le 1 - Fad_{i,m}$$
5. **Evolução da Fadiga Inter-Dias (Bloco E):** Equação de transição de estados de um dia para o outro:
   $$Fad_{i+1, m} = a_m \sum_{j,k} (\alpha_{j,m} \hat{I}_j) Int_{i,j,k} + b_m \cdot Fad_{i,m}$$

In [ ]:
def construir_e_resolver_modelo(nome_cenario, dados):
  dias = dados["dias"]
  slots = dados["slots"]
  disp = dados["disp"]
  musculos = dados["musculos"]
  catalogo = dados["catalogo"]
  a_fad = dados["a_fad"]
  b_fad = dados["b_fad"]
  w_slot = dados["w_slot"]
  I_min = dados.get("I_min", 0.20)
  exercicios = list(catalogo.keys())

  # 1. Pré-processamento dos PRs
  um_rm, I_hat, W_max, utilidade_base = preprocessar_dados(dados)

  # 2. Instanciação do Modelo e Adição Modular das Variáveis
  model = gp.Model(nome_cenario)
  model.Params.OutputFlag = 0

  Ex = adicionar_variavel_Ex(model, dias, exercicios, slots)
  Int = adicionar_variavel_Int(model, dias, exercicios, slots)
  Fad = adicionar_variavel_Fad(model, dias, musculos)
  FadTr = adicionar_variavel_FadTr(model, dias, slots)

  # 3. Função Objetivo
  model.setObjective(
      gp.quicksum(
          w_slot[k] * utilidade_base[j] * Int[i, j, k]
          for i in dias
          for j in exercicios
          for k in slots
      ),
      GRB.MAXIMIZE,
  )

  # 4. Restrições
  for i in dias:
    # Bloco A: Disponibilidade, sem buracos na ordem k e sem repetir exercício no dia
    for k in slots:
      model.addConstr(gp.quicksum(Ex[i, j, k] for j in exercicios) <= disp[i])
    for idx_k in range(len(slots) - 1):
      k_atual, k_prox = slots[idx_k], slots[idx_k + 1]
      model.addConstr(
          gp.quicksum(Ex[i, j, k_prox] for j in exercicios)
          <= gp.quicksum(Ex[i, j, k_atual] for j in exercicios)
      )
    for j in exercicios:
      model.addConstr(gp.quicksum(Ex[i, j, k] for k in slots) <= 1)

    # Bloco B: Acoplamento Exercício-Intensidade (entre 0 e 1) e Queda Intra-Treino
    for j in exercicios:
      for k in slots:
        model.addConstr(Int[i, j, k] >= I_min * Ex[i, j, k])
        model.addConstr(Int[i, j, k] <= Ex[i, j, k])
        model.addConstr(Int[i, j, k] <= 1.0 - FadTr[i, k])

    # Bloco C: Propagação da Fadiga Intra-Treino
    model.addConstr(FadTr[i, slots[0]] == 0.0)
    for idx_k in range(len(slots) - 1):
      k_atual, k_prox = slots[idx_k], slots[idx_k + 1]
      model.addConstr(
          FadTr[i, k_prox]
          == FadTr[i, k_atual]
          + gp.quicksum(
              (catalogo[j]["gamma"] * I_hat[j]) * Int[i, j, k_atual]
              for j in exercicios
          )
      )

    # Bloco D: Capacidade Diária por Grupo Muscular
    for m in musculos:
      model.addConstr(
          gp.quicksum(
              (catalogo[j]["alpha"].get(m, 0.0) * I_hat[j]) * Int[i, j, k]
              for j in exercicios
              for k in slots
          )
          + gp.quicksum(FadTr[i, k] for k in slots)
          <= 1.0 - Fad[i, m]
      )

  # Bloco E: Evolução da Fadiga Inter-Dias
  for m in musculos:
    model.addConstr(Fad[dias[0], m] == 0.0)
    for idx_i in range(len(dias) - 1):
      i_atual, i_prox = dias[idx_i], dias[idx_i + 1]
      model.addConstr(
          Fad[i_prox, m]
          == a_fad[m]
          * gp.quicksum(
              (catalogo[j]["alpha"].get(m, 0.0) * I_hat[j]) * Int[i_atual, j, k]
              for j in exercicios
              for k in slots
          )
          + b_fad[m] * Fad[i_atual, m]
      )

  # 5. Otimização e Exibição de Resultados
  model.optimize()

  print("=" * 85)
  print(f"CENÁRIO: {nome_cenario}")
  if model.status == GRB.OPTIMAL:
    print(f"STATUS: Ótimo Encontrado | Score Hipertrofia: {model.objVal:.2f}")
    print("=" * 85)
    for i in dias:
      status = "DISPONÍVEL" if disp[i] == 1 else "INDISPONÍVEL (Descanso)"
      fad_str = (
          ", ".join([
              f"{m}: {Fad[i, m].X*100:.0f}%"
              for m in musculos
              if Fad[i, m].X > 0.01
          ])
          or "0% (100% Descansado)"
      )
      print(f"\n[DIA {i} - {status}] | Fadiga Muscular Inicial: {fad_str}")

      fez_exercicio = False
      for k in slots:
        for j in exercicios:
          if Ex[i, j, k].X > 0.5:
            fez_exercicio = True
            int_val = Int[i, j, k].X
            peso_kg = W_max[j] * int_val
            reps = catalogo[j]["reps_treino"]
            fad_intra = FadTr[i, k].X
            print(
                f"  -> Slot {k}: {j:<24} | {reps:>2} reps @ {peso_kg:>5.1f} kg "
                f"(Int: {int_val*100:>5.1f}% de {W_max[j]:.1f}kg) | "
                f"FadIntra: {fad_intra*100:>4.1f}%"
            )
      if not fez_exercicio and disp[i] == 1:
        print("  -> [Dia poupado pelo solver devido à fadiga acumulada]")
    print("\n")
  else:
    print(f"Infactível ou Erro. Status Gurobi: {model.status}\n")

### 6. Experimento 1: Mock Simples (Validação de Sanidade)
Neste primeiro teste, avaliamos o comportamento básico do modelo em um horizonte curto de **3 dias** (com o Dia 2 indisponível para descanso), **2 slots por treino** e apenas **2 exercícios de grupos musculares distintos**:
* **Agachamento Livre (`Quadriceps`):** Prioridade alta ($pref = 12$), $PR = 120\text{ kg}$ para $5\text{ reps}$, meta de $8\text{ reps}$ no treino.
* **Supino Reto (`Peito`):** Prioridade moderada ($pref = 8$), $PR = 80\text{ kg}$ para $5\text{ reps}$, meta de $10\text{ reps}$ no treino.

**Comportamento Esperado:** O modelo deve alocar o Agachamento no **Slot 1** (aproveitando $0\%$ de fadiga intra-treino para usar carga máxima) e o Supino no **Slot 2** (sofrendo um leve desconto de carga devido à fadiga intra-treino gerada no Slot 1).

In [ ]:
mock_simples = {
    "dias": [1, 2, 3],
    "slots": [1, 2],
    "disp": {1: 1, 2: 0, 3: 1},
    "musculos": ["Peito", "Quadriceps"],
    "pref": {"Quadriceps": 12.0, "Peito": 8.0},
    "w_slot": {1: 1.0, 2: 0.90},
    "a_fad": {"Peito": 0.60, "Quadriceps": 0.60},
    "b_fad": {"Peito": 0.30, "Quadriceps": 0.30},
    "catalogo": {
        "Agachamento_Livre": {
            "pr_peso": 120.0,
            "pr_reps": 5,
            "reps_treino": 8,
            "gamma": 0.12,
            "alpha": {"Quadriceps": 1.0},
        },
        "Supino_Reto": {
            "pr_peso": 80.0,
            "pr_reps": 5,
            "reps_treino": 10,
            "gamma": 0.10,
            "alpha": {"Peito": 1.0},
        },
    },
}

construir_e_resolver_modelo(
    "MOCK 1 - SIMPLES (2 Exercícios / 3 Dias)", mock_simples
)

### 7. Experimento 2: Mock Complexo (Planejamento Semanal Realista)
Neste segundo cenário, testamos uma rotina completa de **7 dias** (com Quarta-Feira e Domingo indisponíveis), até **4 exercícios por sessão**, **7 grupos musculares** e um catálogo de **10 exercícios** (entre compostos multiarticulares que ativam músculos sinergistas e exercícios isoladores).

**Comportamento Esperado:**
1. **Divisão Automática de Treino (*Split*):** Em dias consecutivos (ex: Dias 1 e 2, ou Dias 4, 5 e 6), o modelo alterna automaticamente os grupos musculares recrutados para respeitar a fadiga local $Fad_{i,m}$.
2. **Hierarquia Intra-Treino:** Exercícios multiarticulares de alta demanda (como Agachamento, Terra Romeno, Supino e Remada) são alocados nos primeiros slots ($k=1, 2$), enquanto exercícios isoladores (Rosca Direta, Tríceps Corda, Extensora, Flexora) preenchem os slots finais ($k=3, 4$).

In [ ]:
mock_complexo = {
    "dias": [1, 2, 3, 4, 5, 6, 7],
    "slots": [1, 2, 3, 4],
    "disp": {1: 1, 2: 1, 3: 0, 4: 1, 5: 1, 6: 1, 7: 0},
    "musculos": [
        "Peito",
        "Costas",
        "Quadriceps",
        "Posterior_Gluteo",
        "Ombro",
        "Biceps",
        "Triceps",
    ],
    "pref": {
        "Quadriceps": 14.0,
        "Posterior_Gluteo": 14.0,
        "Peito": 11.0,
        "Costas": 11.0,
        "Ombro": 8.0,
        "Biceps": 6.0,
        "Triceps": 6.0,
    },
    "w_slot": {1: 1.00, 2: 0.92, 3: 0.85, 4: 0.78},
    "a_fad": {
        m: 0.65
        for m in [
            "Peito",
            "Costas",
            "Quadriceps",
            "Posterior_Gluteo",
            "Ombro",
            "Biceps",
            "Triceps",
        ]
    },
    "b_fad": {
        m: 0.35
        for m in [
            "Peito",
            "Costas",
            "Quadriceps",
            "Posterior_Gluteo",
            "Ombro",
            "Biceps",
            "Triceps",
        ]
    },
    "catalogo": {
        "Agachamento_Livre": {
            "pr_peso": 120.0,
            "pr_reps": 5,
            "reps_treino": 6,
            "gamma": 0.08,
            "alpha": {"Quadriceps": 1.0, "Posterior_Gluteo": 0.5},
        },
        "Terra_Romeno_RDL": {
            "pr_peso": 110.0,
            "pr_reps": 6,
            "reps_treino": 8,
            "gamma": 0.08,
            "alpha": {"Posterior_Gluteo": 1.0, "Costas": 0.3},
        },
        "Cadeira_Extensora": {
            "pr_peso": 75.0,
            "pr_reps": 10,
            "reps_treino": 12,
            "gamma": 0.04,
            "alpha": {"Quadriceps": 1.0},
        },
        "Mesa_Flexora": {
            "pr_peso": 65.0,
            "pr_reps": 10,
            "reps_treino": 12,
            "gamma": 0.04,
            "alpha": {"Posterior_Gluteo": 1.0},
        },
        "Supino_Reto_Barra": {
            "pr_peso": 85.0,
            "pr_reps": 5,
            "reps_treino": 8,
            "gamma": 0.07,
            "alpha": {"Peito": 1.0, "Ombro": 0.4, "Triceps": 0.5},
        },
        "Desenvolvimento_Halter": {
            "pr_peso": 52.0,
            "pr_reps": 8,
            "reps_treino": 10,
            "gamma": 0.06,
            "alpha": {"Ombro": 1.0, "Triceps": 0.4},
        },
        "Triceps_Corda": {
            "pr_peso": 35.0,
            "pr_reps": 10,
            "reps_treino": 12,
            "gamma": 0.03,
            "alpha": {"Triceps": 1.0},
        },
        "Remada_Curvada": {
            "pr_peso": 80.0,
            "pr_reps": 6,
            "reps_treino": 8,
            "gamma": 0.07,
            "alpha": {
                "Costas": 1.0,
                "Biceps": 0.5,
                "Posterior_Gluteo": 0.2,
            },
        },
        "Puxada_Frontal": {
            "pr_peso": 70.0,
            "pr_reps": 8,
            "reps_treino": 10,
            "gamma": 0.05,
            "alpha": {"Costas": 1.0, "Biceps": 0.4},
        },
        "Rosca_Direta_Barra": {
            "pr_peso": 36.0,
            "pr_reps": 8,
            "reps_treino": 12,
            "gamma": 0.03,
            "alpha": {"Biceps": 1.0},
        },
    },
}

construir_e_resolver_modelo(
    "MOCK 2 - COMPLEXO (10 Exercícios / 7 Dias)", mock_complexo
)

# 8. Conclusão e Considerações Finais

### Síntese da Evolução do Modelo
Ao longo deste projeto, partimos de uma formulação contínua macroscópica (baseada em um volume agregado de treino diário $Tr_i$ e fadiga sistêmica única $Fad_i$) e evoluímos para um modelo de **Programação Linear Inteira Mista (MILP)** capaz de prescrever uma rotina completa e fisiologicamente coerente de musculação. 

As principais contribuições de modelagem alcançadas foram:
1. **Acoplamento Linear entre Decisão Discreta e Carga Contínua ($Ex_{i,j,k}$ e $Int_{i,j,k}$):** Utilizando restrições de variáveis semi-contínuas ($I_{\min} \cdot Ex_{i,j,k} \le Int_{i,j,k} \le Ex_{i,j,k}$) combinadas com o pré-processamento de *Personal Records* (via equação de Epley), o modelo prescreve simultaneamente **qual exercício fazer** e **qual carga exata (em kg) utilizar**, mantendo a formulação $100\%$ linear e garantindo otimalidade global no Gurobi.
2. **Emergência Natural de Divisões de Treino (*Splits*):** Ao desacoplar a fadiga diária por grupo muscular ($Fad_{i,m}$) e ponderar a ativação de músculos agonistas e sinergistas ($\alpha_{j,m}$), eliminamos a necessidade de pré-fixar rótulos rígidos de treino (como *Push/Pull/Legs* ou *Full Body*). O próprio solver adapta a divisão ideal à disponibilidade semanal do praticante ($Disp_i$).
3. **Hierarquia Intra-Treino e Ordenação Contígua ($k$):** A propagação da fadiga intra-treino ($FadTr_{i,k}$) aliada ao fator de eficiência neuromuscular por slot ($w_k$) reproduziu fielmente a prática esportiva: exercícios multiarticulares de alta demanda são alocados no início da sessão ($k=1, 2$), enquanto exercícios isoladores preenchem os slots finais com ajuste automático de carga.

### Trabalhos Futuros e Extensões Possíveis
Como próximos passos para expandir a aplicação prática deste modelo, destacam-se:
* **Periodização Multi-Semanas (Mesociclos):** Expandir o horizonte $D$ para 4 a 8 semanas, incorporando sobrecarga progressiva e semanas de *deload* (redução programada de volume/intensidade para supercompensação).
* **Decisão do Número de Séries e Tempo de Sessão:** Associar uma variável inteira ao número de séries por exercício limitada pelo tempo máximo disponível do usuário em cada dia (em minutos).
* **Restrições de Equilíbrio Articular/Postural:** Incluir restrições de proporção mínima entre cadeias musculares antagonistas (ex: razão entre volume de *Puxar* e *Empurrar*, ou *Quadríceps* e *Posterior de Coxa*) para prevenção de lesões.